# Notebook 4: Simple Direct Adapter Test UI

Bu notebook Notebook 3'e dokunmadan daha basit bir kullanici akisi sunar.

Kullanim:
1. Ilk kod hucresini bir kez calistirin.
2. Ikinci kod hucresini bir kez calistirin.
3. Acilan arayuzden adapter secin, resmi yukleyin ve `Tahmin Et` butonuna basin.
4. Yeni tahmin icin hucreyi tekrar calistirmayin; panelde `Yeni Resim` veya `Resim Yukle` ile gorseli degistirip tekrar `Tahmin Et` basin.


## Hizli Akis

Bu notebook Notebook 3'un basit butonlu arayuzudur; adapter secimi ve tek goruntu tahmini icin tasarlanmistir.

- Ilk hucresi repo ve erisim kontrolunu hazirlar, ikinci hucresi UI panelini acar.
- Panel acildiktan sonra yeni tahmin icin hucreleri tekrar calistirmayin; panelde resmi degistirip tekrar `Tahmin Et` basin.
- Daha fazla tanisal cikti, batch test veya explanation map gerekiyorsa Notebook 3'e gecin.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import tempfile
import urllib.request
import shutil

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
# Set to '1' or 'true' to allow cloning (default: enabled)
ALLOW_CLONE = os.environ.get('AADS_ALLOW_CLONE', '1') in ('1', 'true', 'True')
ALLOW_CLONE = os.environ.get('AADS_ALLOW_CLONE', '0') in ('1', 'true', 'True')

def _find_repo_root(start: Path = None):
    start = start or Path.cwd()
    for p in [start] + list(start.parents):
        if (p / 'scripts').is_dir() and (p / 'src').is_dir():
            return p.resolve()
    return None

def _download_file(raw_base, rel_path, dest_root):
    dest_path = Path(dest_root) / rel_path
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    url = f'{raw_base}/{rel_path}'
    print(f'Downloading {url} -> {dest_path}')
    with urllib.request.urlopen(url) as r, open(dest_path, 'wb') as f:
        f.write(r.read())
    return dest_path

    # If configured to clone, perform cloning first and skip local search
    if ALLOW_CLONE:
        if not CLONE_TARGET.exists():
            print(f"Cloning repository to {CLONE_TARGET}...")
            subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_TARGET)], check=True)
        if str(CLONE_TARGET) not in sys.path:
            sys.path.insert(0, str(CLONE_TARGET))
        print(f"✓ Repo cloned/accessed: {CLONE_TARGET}")
        return CLONE_TARGET
    # Try common candidates first
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        CLONE_TARGET,
        Path('/content/bitirmeprojesi'),
        Path('/content/bitirme projesi')
    ]
    for candidate in candidates:
        marker = candidate / 'scripts' / 'notebook_helpers' / 'cell_script_runner.py'
        if marker.is_file():
            repo_root = candidate.resolve()
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            print(f"✓ Repo root detected via candidates: {repo_root}")
            return repo_root

    # Walk parents as a more robust fallback
    repo_root = _find_repo_root(Path.cwd())
    if repo_root:
        if str(repo_root) not in sys.path:
            sys.path.insert(0, str(repo_root))
        print(f"✓ Repo root detected by walking parents: {repo_root}")
        return repo_root

    # If cloning explicitly allowed, clone as last resort
    if ALLOW_CLONE:
        if not CLONE_TARGET.exists():
            print(f"Cloning repository to {CLONE_TARGET}...")
            subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_TARGET)], check=True)
        if str(CLONE_TARGET) not in sys.path:
            sys.path.insert(0, str(CLONE_TARGET))
        print(f"✓ Repo cloned/accessed: {CLONE_TARGET}")
        return CLONE_TARGET

    # Otherwise fetch minimal helper files from GitHub raw
    print('Repo not found locally and cloning disabled. Fetching minimal helper scripts from GitHub raw...')
    repo = REPO_URL.rstrip('.git').rstrip('/')
    if repo.startswith('https://github.com/'):
        raw_base = repo.replace('https://github.com/', 'https://raw.githubusercontent.com/') + '/main'
    else:
        raise RuntimeError('Unsupported REPO_URL format for raw fetch: ' + REPO_URL)

    temp_dir = Path(tempfile.mkdtemp(prefix='aads_min_'))
    print(f'Using temp dir: {temp_dir}')
    files_to_fetch = [
        'scripts/notebook_helpers/cell_script_runner.py',
        'scripts/colab_simple_adapter_smoke_ui.py',
    ]
    for rel in files_to_fetch:
        try:
            _download_file(raw_base, rel, temp_dir)
        except Exception as e:
            shutil.rmtree(temp_dir)
            raise RuntimeError(f'Failed to download {rel}: {e}') from e

    if str(temp_dir) not in sys.path:
        sys.path.insert(0, str(temp_dir))
    print(f"✓ Minimal helpers downloaded to {temp_dir}")
    return temp_dir

repo_root = _ensure_aads_repo_on_path()

# Import bootstrap helpers (fallback to loading by path if needed)
try:
    from scripts.notebook_helpers.cell_script_runner import run_cell_script
except Exception:
    import importlib.util









run_cell_script('nb4_cell01_bootstrap_access.py', globals())# Run the bootstrap cell scriptprint(f"✓ sys.path[0]: {sys.path[0]}")    run_cell_script = module.run_cell_script    spec.loader.exec_module(module)    module = importlib.util.module_from_spec(spec)    spec = importlib.util.spec_from_file_location('cell_script_runner', str(Path(repo_root)/'scripts'/'notebook_helpers'/'cell_script_runner.py'))

In [ ]:
import importlib
from scripts import colab_simple_adapter_smoke_ui

colab_simple_adapter_smoke_ui = importlib.reload(colab_simple_adapter_smoke_ui)
launch_simple_adapter_smoke_ui = colab_simple_adapter_smoke_ui.launch_simple_adapter_smoke_ui

print('[UI] Adapter secimi ve tek goruntu tahmini paneli aciliyor. Panel acildiktan sonra yeni tahminler panel icinden yapilir.')
launch_simple_adapter_smoke_ui(ROOT, show_all_adapters=True, show_mirror_adapters=True)
